# MirrorTopology Step 1 — Phase D4C-2a：**count-envelope screen**（成功不能証明の十分条件）launcher（notebook v0.1）
1 run＝1 family（E2／E7／E8；E1 は position 分岐免除で対象外）：受入れ済み D-2 run の第 1 波 unit だけを formal intake し（D-3b・12 位置入力・bootstrap・KDE は使わない），登録 D-2W W₂ context の判定（case ごとの True／False／unknown）と登録 pseudo 列（n＝2000）を registered loader から取り，`step1_engine.infeasibility.envelope_screen_family` を実行する：各 pseudo 行・各 size について matched system の 3 position × 両 prefix（N0／N4）の **正確な hit 率** P＝point_P(hits, N)（`orchestrator._family_Q` と同じ float64 計算）を数え，全 hit＞0 かつ max／min ≤ 2.0（登録 event-ratio trigger は max／min＞2.0）なら，その size はどの N 選択でも position-sensitive になれない。family に W₂ unknown の size が 1 つ以上あり W₂ True の size がなければ，全 size が条件を満たす行の eligible truth は（技術 FAIL を除き）**unknown にしかなれない**（blocking 行）。条件を満たさない行は「未判定」であり False とは言わない。出力は `d4c1_screen_<F>[_r<start>_<stop>]_record.json`（schema `position_envelope_screen_v1`；hit count・P・ratio bound・W₂ 判定・列 identity・fingerprint を記録）。**これは較正ではない**（c／u／n・Wilson・usable・label を発行しない；partial／sealed の reader は受け付けない）；証明書（certificate）は screen record と評価済み record から別 mode（`--mode certificate`）で作る（Colab 不要）。標準 runtime で可（第 1 波 18 配置 ≈4 GB＋plan ≈3.4 GB）。

**attempt cell の規約**は partial launcher と同一（anchor・1 つの検査関数を staging 前と起動直前の 2 回・attempt 別 initial 非成功 record・信頼 REQUIRED inventory（`REQUIRED_SCREEN` を AST 抽出）・published evidence の内容束縛（JSON object でなければ失敗））。`screen_complete` は rc 0 ∧ `D4C1_SCREEN_COMPLETE` ∧ shape 正常 ∧ 全 gate True ∧ attempt／lock／source／family／commitment／campaign の束縛 ∧ published screen record の SHA／bytes 一致と内容束縛（schema・family・pseudo n 2000＋列 SHA＝登録定数・context SHA＝pins・rows・screen SHA）のときだけ True。科学的結果（blocking 行数）は条件ではない。


In [1]:
# --- 0. OUTER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = 'ba51e5951ebb8c16c7106247408c912d86944675'
EXPECTED_INVENTORY_SHA256 = 'aa2c11cc01df09fc9cd130a2c0b9aa5af75d1b2acc27f47aef3d5d91d0bb2b77'
FAMILY = 'E2'                                                                   # E2 / E7 / E8 (E1 is position-branch exempt: no screen)
D2_RUN_ROOT = '/content/drive/MyDrive/MirrorTopology_D2/E2_20260923T092030Z/out/d2'                 # the accepted D-2 run of FAMILY (out/d2); first-wave units only are read
ROW_RANGE = None
TARGET_COMMITMENT = '9efa1b8d82fec2e23ac685d24dbaece7932e29955c94fc5a2955ba793c70b326'
CAMPAIGN_ID = 'D4C2_9efa1b8d82fe'
OUT_ROOT = '/content/drive/MyDrive/MirrorTopology_D4C2_screen_0_112_0'
STAGE_LOCAL = True
STAGE_ROOT = '/content/stage'

In [2]:
# --- 1. fresh scratch checkout; inventory + script + pins + registered ledgers / receipts / assets bound; the D-2 input exists and matches the ledger; Drive mounted (read-only usage); the LOCK (published, hashed)
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and re.fullmatch(r'[0-9a-f]{64}', TARGET_COMMITMENT) and isinstance(CAMPAIGN_ID, str) and len(CAMPAIGN_ID)>=8 and '<' not in CAMPAIGN_ID
assert FAMILY in ('E2','E7','E8'), 'the screen applies to the position-branch families only'
assert ROW_RANGE is None or (isinstance(ROW_RANGE,(tuple,list)) and len(ROW_RANGE)==2 and all(isinstance(x,int) and not isinstance(x,bool) for x in ROW_RANGE) and 0<=ROW_RANGE[0]<ROW_RANGE[1]<=2000), 'ROW_RANGE must be None or (start, stop) with 0 <= start < stop <= 2000'
assert OUT_ROOT is None or (isinstance(OUT_ROOT,str) and OUT_ROOT.startswith('/content/drive/MyDrive/') and '<' not in OUT_ROOT and OUT_ROOT==OUT_ROOT.rstrip('/')), 'OUT_ROOT must be None or a Drive folder under /content/drive/MyDrive'
SUB=ROW_RANGE is not None; RTAG=(f'_r{ROW_RANGE[0]:04d}_{ROW_RANGE[1]:04d}' if SUB else '')
from google.colab import drive; drive.mount('/content/drive')
assert os.path.isfile(f'{D2_RUN_ROOT}/d2_bank_registry.json'), D2_RUN_ROOT
SCRATCH='/content/d4c1_scratch'; shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip()==REPO_COMMIT and subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
MT=SCRATCH; PHASEB=f'{SCRATCH}/engine/phaseB'; PHASEC=f'{SCRATCH}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; assert sha(INV)==EXPECTED_INVENTORY_SHA256; inv=json.load(open(INV)); SCRIPT=f'{PHASEB}/d/d4c1_calibration.py'; assert sha(SCRIPT)==inv['d_sha256']['d/d4c1_calibration.py']
RA={k: f'{PHASEB}/{k}' for k in ('registered_assets/d2/d2_generation_ledger.json','registered_assets/d3b/d3b_generation_ledger.json','registered_assets/d3b/d3b_outer_receipt.json','registered_assets/d3c/d3c_profile_ledger.json','registered_assets/d3c/d3c_outer_receipt.json','registered_assets/d4c0/d4c0_ledger.json','registered_assets/d4c0/d4c0_outer_receipt.json','registered_assets/b3_2_twelve_assets.json','registered_assets/b3_2_shared_null_asset.json')}
PINS=f'{PHASEB}/d/d3_pins.json'; assert sha(PINS)==inv['d_sha256']['d/d3_pins.json'] and all(sha(p)==inv['registered_assets_sha256'][k] for k,p in RA.items()); RA_SHA={k: sha(p) for k,p in RA.items()}
d2l=json.load(open(RA['registered_assets/d2/d2_generation_ledger.json'])); assert D2_RUN_ROOT.rstrip('/')==d2l['families'][FAMILY]['run_root'].rstrip('/')+'/out/d2', (D2_RUN_ROOT, d2l['families'][FAMILY]['run_root'])
pins=json.load(open(PINS)); ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl','psutil'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
import psutil; vm=psutil.virtual_memory(); RAM_INFO=dict(total_bytes=vm.total, available_bytes=vm.available); print('RAM total GB', round(vm.total/1e9,1), 'available GB', round(vm.available/1e9,1)); assert vm.total>10e9, 'the screen needs > 10 GB RAM (first-wave intake + plans)'
OUT=f'{OUT_ROOT or "/content"}/d4c1_screen_{FAMILY}{RTAG}'
# OUTPUT-ANCHOR (R-D4C0-B; the lines from here to the end of this cell are re-used verbatim by the contract test): the output directory is validated BEFORE the first mkdir / lock publication
_in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
OWNED=re.compile(r'(d4c1_screen_lock\.json|d4c1_screen_final_record\.json|superseded_final_record_before_[0-9TZ]+_[0-9a-f]{10}\.json|run_[0-9TZ]+_[0-9a-f]{10}|d4c1_screen_(stdout|stderr)_[0-9TZ]+_[0-9a-f]{10}\.txt)')
def _protected_roots(): return [MT, PHASEB, PHASEC, STAGE_ROOT, D2_RUN_ROOT]+([] if OUT_ROOT else ['/content/drive'])
def _safe_output_anchor(path, protected):
    # the ONE output policy (preflight and attempt): absolute path whose final component is not a symlink; disjoint from every protected root (source, Phase C, stage, Drive, inputs);
    # non-existent, or an existing real directory holding ONLY files owned by this launcher (lock, attempt records, run_* directories, stdout/stderr) and no symlinks. Nothing is written here.
    ap=os.path.abspath(path)
    if os.path.islink(ap) or os.path.realpath(ap)!=os.path.join(os.path.realpath(os.path.dirname(ap)), os.path.basename(ap)): raise RuntimeError(f'unsafe output: {path!r} is a symbolic link; nothing written')
    clash=[q for q in protected if _in(ap,q) or _in(q,ap)]
    if clash: raise RuntimeError(f'unsafe output: {path!r} overlaps a protected root {clash}; nothing written')
    if os.path.lexists(ap):
        if not os.path.isdir(ap): raise RuntimeError(f'unsafe output: {path!r} exists and is not a directory; nothing written')
        foreign=[e for e in sorted(os.listdir(ap)) if not OWNED.fullmatch(e) or os.path.islink(os.path.join(ap,e))]
        if foreign: raise RuntimeError(f'unsafe output: {path!r} holds entries not owned by this launcher {foreign[:5]}; nothing written')
    return ap
OUT=_safe_output_anchor(OUT, _protected_roots()); os.makedirs(OUT, exist_ok=True)      # attempts are separated inside OUT by their attempt id (nothing is deleted here)
# LOCK-BUILD
def _publish(path, doc):
    data=json.dumps(doc, indent=1, ensure_ascii=False, default=str).encode('utf-8'); tmp=path+'.tmp'; open(tmp,'wb').write(data); os.replace(tmp, path); assert open(path,'rb').read()==data, path; return hashlib.sha256(data).hexdigest()
lock=dict(schema='d4c1_screen_launcher_lock_v1', commit=REPO_COMMIT, inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv['d_sha256']['d/d4c1_calibration.py'], pins_sha256=sha(PINS), registered_assets_sha256=RA_SHA, engine=inv['engine_version'],
          family=FAMILY, d2_run_root=D2_RUN_ROOT, row_range=(list(ROW_RANGE) if SUB else None), target_commitment=TARGET_COMMITMENT, campaign_id=CAMPAIGN_ID, out_root=OUT_ROOT, stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT, locked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
LOCK_PATH=f'{OUT}/d4c1_screen_lock.json'; LOCK_SHA256=_publish(LOCK_PATH, lock); print('lock', LOCK_SHA256); print({k: v for k, v in lock.items() if k != 'registered_assets_sha256'})


Mounted at /content/drive
RAM total GB 54.8 available GB 52.6
lock ac223b5ddb1e964edc796edcf0049f7fb3e255063e7b26691d354e314cf463dc
{'schema': 'd4c1_screen_launcher_lock_v1', 'commit': 'ba51e5951ebb8c16c7106247408c912d86944675', 'inventory_sha256': 'aa2c11cc01df09fc9cd130a2c0b9aa5af75d1b2acc27f47aef3d5d91d0bb2b77', 'script_sha256': '814557791d7f295fbb981c20806c7374ddf7ae19148380c41c004afef3e597b0', 'inventory_script_sha256': '814557791d7f295fbb981c20806c7374ddf7ae19148380c41c004afef3e597b0', 'pins_sha256': '1d03bc5f83d41a2bf75d817cf7d6b9d29caf856740a4f87dc9e8d303a7e172d2', 'engine': '0.112.0', 'family': 'E2', 'd2_run_root': '/content/drive/MyDrive/MirrorTopology_D2/E2_20260923T092030Z/out/d2', 'row_range': None, 'target_commitment': '9efa1b8d82fec2e23ac685d24dbaece7932e29955c94fc5a2955ba793c70b326', 'campaign_id': 'D4C2_9efa1b8d82fe', 'out_root': '/content/drive/MyDrive/MirrorTopology_D4C2_screen_0_112_0', 'stage_local': True, 'stage_root': '/content/stage', 'mt': '/content/d4c1_scratc

In [3]:
# --- 2. ONE ATTEMPT (partial launcher pattern): (0) the SAFE anchored output is taken from the LOCK and validated BEFORE anything is written; (1) attempt id + initial NON-success record; (2) ONE verifier BEFORE staging;
#        (3) staging of the first-wave D-2 units (disk check first); (4) the SAME verifier IMMEDIATELY before subprocess.run; (5) record validation (trusted REQUIRED_SCREEN) + binding + published screen record content AND semantic check
t0=time.time(); _in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
ANCHOR=lock['out']
try:
    if '_safe_output_anchor' not in globals(): raise RuntimeError('preflight cell not run')
    _safe_output_anchor(ANCHOR, [lock['mt'], lock['phaseb'], lock['phasec'], lock['stage_root'], lock['d2_run_root'], *_protected_roots()])
    if not os.path.isdir(ANCHOR) or LOCK_PATH!=f'{ANCHOR}/d4c1_screen_lock.json' or not os.path.isfile(LOCK_PATH): raise RuntimeError('lock path not inside the anchor')
except Exception as _anchor_error: print(f'ATTEMPT NOT STARTED: the locked output {ANCHOR!r} is not a safe anchored directory ({_anchor_error}); nothing was written', file=sys.stderr); raise RuntimeError('unsafe output anchor; nothing written')
ATTEMPT=time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())+'_'+hashlib.sha256(f'{time.time_ns()}|D4C2S|{REPO_COMMIT}|{LOCK_SHA256}'.encode()).hexdigest()[:10]
FIN=f'{ANCHOR}/d4c1_screen_final_record.json'; RUN=f'{ANCHOR}/run_{ATTEMPT}'; screen_complete=False; rc=None; v=None; RR=lock['row_range']; SUB=RR is not None; TAG=f"screen_{lock['family']}"+(f'_r{RR[0]:04d}_{RR[1]:04d}' if SUB else '')
REC=dict(schema='d4c1_screen_launcher_final_record_v1', attempt_id=ATTEMPT, started_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), lock_sha256=LOCK_SHA256, lock=lock, output_anchor=ANCHOR, ram=globals().get('RAM_INFO'), stage='started', screen_complete=False, row_range=RR, exit_code=None, D4C1_SCREEN_COMPLETE=None, launcher_fallback=False, gates_ok=None, bindings_ok=None, evidence_ok=None, failures=[], exception=None, run_dir=RUN, staged_inputs=None, bindings={}, seconds_total=None, finished_utc=None)
if os.path.exists(FIN): prev=f'{ANCHOR}/superseded_final_record_before_{ATTEMPT}.json'; os.replace(FIN, prev); REC['superseded_previous_record']=prev
_publish(FIN, REC)
def _finish(stage, failures=(), exc=None):
    REC.update(stage=stage, exception=(repr(exc) if exc is not None else None), seconds_total=round(time.time()-t0, 3), finished_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())); REC['failures']+=list(failures); REC['screen_complete']=bool(screen_complete); _publish(FIN, REC); print('attempt', ATTEMPT, '| stage', stage, '| screen_complete', REC['screen_complete'], '| failures', REC['failures'])
def _verify_locked_state(prefix):
    REC['stage']=f'{prefix}_config'; cfg=dict(family=FAMILY, d2_run_root=D2_RUN_ROOT, row_range=(list(ROW_RANGE) if ROW_RANGE is not None else None), target_commitment=TARGET_COMMITMENT, campaign_id=CAMPAIGN_ID, out_root=OUT_ROOT, stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, commit=REPO_COMMIT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT)
    mism={k: (cfg[k], lock.get(k)) for k in cfg if cfg[k]!=lock.get(k)}
    if mism: raise RuntimeError(f'configuration differs from the lock (re-run the preflight cell): {mism}')
    if LOCK_PATH!=f'{ANCHOR}/d4c1_screen_lock.json' or sha(LOCK_PATH)!=LOCK_SHA256 or json.load(open(LOCK_PATH))!=lock: raise RuntimeError('lock file on disk differs from the lock in memory')
    REC['stage']=f'{prefix}_source'; inv_now=json.load(open(INV))
    src=dict(head=subprocess.check_output(['git','-C',MT,'rev-parse','HEAD'],text=True).strip(), clean=(subprocess.check_output(['git','-C',MT,'status','--porcelain'],text=True).strip()==''), inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv_now['d_sha256']['d/d4c1_calibration.py'], pins_sha256=sha(PINS), registered_assets_sha256={k: sha(p) for k,p in RA.items()}, engine=inv_now['engine_version'], checked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
    want=dict(head=lock['commit'], clean=True, inventory_sha256=lock['inventory_sha256'], script_sha256=lock['script_sha256'], inventory_script_sha256=lock['script_sha256'], pins_sha256=lock['pins_sha256'], registered_assets_sha256=lock['registered_assets_sha256'], engine=lock['engine'])
    bad=[k for k in want if src[k]!=want[k]]; REC['bindings'][f'live_source_{prefix}']=src
    if bad or PHASEB!=f'{MT}/engine/phaseB' or PHASEC!=f'{MT}/phaseC' or SCRIPT!=f'{PHASEB}/d/d4c1_calibration.py': raise RuntimeError(f'live source differs from the lock ({prefix}): {bad}')
    if not os.path.isdir(D2_RUN_ROOT): raise RuntimeError(f'input root missing: {D2_RUN_ROOT}')
try:
    _verify_locked_state('precheck'); _publish(FIN, REC)
    REC['stage']='staging'; IN_D2=D2_RUN_ROOT
    if STAGE_LOCAL:
        ST=STAGE_ROOT; guard=[q for q in [D2_RUN_ROOT, ANCHOR, MT] if _in(ST,q) or _in(q,ST)]
        if guard: raise RuntimeError(f'STAGE_ROOT must be disjoint from the inputs, the output and the source: {guard}')
        units=sorted(u for u in os.listdir(D2_RUN_ROOT) if (re.fullmatch(r'cfg\d+_(b0|b1|fit)', u) or re.fullmatch(r'ref_'+FAMILY+r'_(b0|b1|fit)', u)) and os.path.isdir(os.path.join(D2_RUN_ROOT,u)))
        dsize=lambda root: sum(os.path.getsize(os.path.join(d,x)) for d,_,fs in os.walk(root) for x in fs)
        need=sum(dsize(os.path.join(D2_RUN_ROOT,u)) for u in units)+os.path.getsize(os.path.join(D2_RUN_ROOT,'d2_bank_registry.json')); ts=time.time()
        shutil.rmtree(ST, ignore_errors=True); os.makedirs(ST); free=shutil.disk_usage(ST).free; REC['staging']=dict(root=ST, d2_units=len(units), input_bytes=need, disk_free_bytes_before=free)
        if free < need*1.05+2e9: raise RuntimeError(f'insufficient local disk for staging: need {need/1e9:.1f} GB (+5% +2 GB margin), free {free/1e9:.1f} GB')
        IN_D2=f'{ST}/d2'; os.makedirs(IN_D2); shutil.copy2(os.path.join(D2_RUN_ROOT,'d2_bank_registry.json'), os.path.join(IN_D2,'d2_bank_registry.json'))
        for u in units: shutil.copytree(os.path.join(D2_RUN_ROOT,u), os.path.join(IN_D2,u))
        REC['staging'].update(seconds=round(time.time()-ts,1), disk_free_bytes_after=shutil.disk_usage(ST).free); print('staged in', round(time.time()-ts), 's; disk free GB', round(REC['staging']['disk_free_bytes_after']/1e9,1))
    REC['staged_inputs']=dict(d2=IN_D2); _publish(FIN, REC)
    _verify_locked_state('prelaunch')
    REC['stage']='launch'; args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--out',RUN,'--mode','screen','--family',FAMILY,'--d2-root',IN_D2,'--target-commitment',TARGET_COMMITMENT,'--campaign-id',CAMPAIGN_ID,'--profile','production_official','--attempt-id',ATTEMPT,'--launcher-lock-sha256',LOCK_SHA256]+([] if not SUB else ['--row-range', f'{RR[0]}:{RR[1]}'])
    REC['args']=args; _publish(FIN, REC); rc=subprocess.run(args,capture_output=True,text=True); REC['exit_code']=rc.returncode
    open(f'{ANCHOR}/d4c1_screen_stdout_{ATTEMPT}.txt','w').write(rc.stdout); open(f'{ANCHOR}/d4c1_screen_stderr_{ATTEMPT}.txt','w').write(rc.stderr); print(rc.stdout[-3000:]); print('rc', rc.returncode)
    REC['stage']='record'; vp=f'{RUN}/d4c1_{TAG}_run.json'; shape=[]; gates_bad=[]; binding=[]; evidence=[]
    REQUIRED_EXPECTED=['G_pins_loaded','G_engine_inventory','G_script_sha','G_phaseC_members','G_env_lock','G_external_loader_sha','G_twelve_context','G_w2_context_registered','G_pseudo_registered','G_twelve_asset_registered','G_commitment_form','G_inputs_resolved','G_first_wave_supplies','G_plans_fixed','G_first_wave_views','G_row_range','G_screen_computed','G_screen_published','G_pseudo_identity_bound','G_record_saved']
    import ast as _ast
    _req_src=[list(_ast.literal_eval(st.value)) for st in _ast.parse(open(SCRIPT).read()).body if isinstance(st,_ast.Assign) and any(isinstance(t,_ast.Name) and t.id=='REQUIRED_SCREEN' for t in st.targets)]
    if _req_src!=[REQUIRED_EXPECTED]: gates_bad.append(f'REQUIRED_SCREEN inventory of the lock-bound script differs from the trusted constant: {_req_src}')
    try: v=json.load(open(vp))
    except Exception as read_error: v=None; shape.append(f'run record unreadable: {read_error!r}')
    if v is not None and not isinstance(v, dict): shape.append('run record is not a dictionary'); v=None
    if v is not None:
        num=lambda x: isinstance(x,(int,float)) and not isinstance(x,bool)
        for k, ok in [('schema', v.get('schema')=='d4c1_run_record_v1'), ('mode', v.get('mode')=='screen'), ('family', v.get('family')==FAMILY), ('D4C1_SCREEN_COMPLETE', isinstance(v.get('D4C1_SCREEN_COMPLETE'), bool)), ('D4C1_PARTIAL_PASS', v.get('D4C1_PARTIAL_PASS') is False), ('row_range', v.get('row_range')==RR), ('stage', v.get('stage')=='complete'), ('gates', isinstance(v.get('gates'), dict)), ('required_inventory', isinstance(v.get('required_inventory'), list)), ('required_all_true', isinstance(v.get('required_all_true'), bool)), ('failures', v.get('failures')==[]), ('seconds', num(v.get('seconds'))),
                      ('attempt', isinstance(v.get('attempt'), dict)), ('source', isinstance(v.get('source'), dict)), ('selftest', v.get('selftest') is False), ('formal', v.get('formal') is True), ('probe', v.get('probe') is False), ('profile', v.get('profile')=='production_official'), ('screen', isinstance(v.get('screen'), dict)), ('pseudo', isinstance(v.get('pseudo'), dict)), ('w2_context', isinstance(v.get('w2_context'), dict)), ('published_evidence', isinstance(v.get('published_evidence'), dict))]:
            if not ok: shape.append(f'run record field invalid: {k}')
        g=v.get('gates') if isinstance(v.get('gates'), dict) else {}
        if sorted(g)!=sorted(REQUIRED_EXPECTED): gates_bad.append(f'gate set differs from the trusted REQUIRED inventory: {sorted(g)}')
        if v.get('required_inventory')!=REQUIRED_EXPECTED: gates_bad.append('required_inventory differs from the trusted REQUIRED inventory')
        if not all(g.get(k) is True for k in REQUIRED_EXPECTED): gates_bad.append(f'required gates not all True: {[k for k in REQUIRED_EXPECTED if g.get(k) is not True]}')
        if v.get('required_all_true') is not True: gates_bad.append('required_all_true is not True')
        a_=v.get('attempt') if isinstance(v.get('attempt'), dict) else {}; s_=v.get('source') if isinstance(v.get('source'), dict) else {}
        for k, ok in [('attempt_id', a_.get('attempt_id')==ATTEMPT), ('launcher_lock_sha256', a_.get('launcher_lock_sha256')==LOCK_SHA256), ('source.script_sha256', s_.get('script_sha256')==lock['script_sha256']), ('source.inventory_sha256', s_.get('inventory_sha256')==lock['inventory_sha256']), ('source.pins_sha256', s_.get('pins_sha256')==lock['pins_sha256']), ('source.engine_version', s_.get('engine_version')==lock['engine']), ('target_commitment', v.get('target_commitment')==lock['target_commitment']), ('campaign_id', v.get('campaign_id')==lock['campaign_id'])]:
            if not ok: binding.append(f'run record not bound to this attempt / lock: {k}')
        pe=v.get('published_evidence') if isinstance(v.get('published_evidence'), dict) else {}; fn=f'd4c1_{TAG}_record.json'; fp=f'{RUN}/{fn}'; e=pe.get(fn); ps_=v.get('pseudo') if isinstance(v.get('pseudo'), dict) else {}; idn=ps_.get('identity') if isinstance(ps_.get('identity'), dict) else {}; w2_=v.get('w2_context') if isinstance(v.get('w2_context'), dict) else {}; sc=v.get('screen') if isinstance(v.get('screen'), dict) else {}
        if set(pe)!={fn}: evidence.append(f'published evidence set {sorted(pe)} != {[fn]}')
        if not (isinstance(e, dict) and os.path.isfile(fp) and not os.path.islink(fp) and sha(fp)==e.get('sha256') and os.path.getsize(fp)==e.get('bytes')): evidence.append(f'published screen record not content-verified: {fn}')
        else:
            try: doc=json.load(open(fp)); doc_ok=True
            except Exception as doc_error: doc=None; doc_ok=False; evidence.append(f'published screen record unreadable: {doc_error!r}')
            if doc_ok and not isinstance(doc, dict): evidence.append(f'published screen record is not a JSON object ({type(doc).__name__}); its content cannot be bound')      # R-D4C1-B
            if doc_ok and isinstance(doc, dict):
                pp=doc.get('pseudo') if isinstance(doc.get('pseudo'), dict) else {}; rows_expected=list(range(*RR)) if SUB else list(range(2000)); b=doc.get('binding') if isinstance(doc.get('binding'), dict) else {}
                checks=[('schema', doc.get('schema')=='position_envelope_screen_v1'), ('family', doc.get('family')==FAMILY), ('engine', doc.get('engine_version')==lock['engine']), ('commitment', doc.get('target_commitment')==lock['target_commitment']), ('pseudo', pp.get('n')==2000 and pp.get('sha256_T1')==idn.get('T1_sha256') and pp.get('sha256_T2')==idn.get('T2_sha256') and idn.get('paired_sha256')==pins['pseudo_paired_sha256']),
                        ('context_sha', doc.get('w2_context_sha256')==pins['d2w_context_sha256']==w2_.get('context_sha256')), ('rows', doc.get('rows')==rows_expected and doc.get('n_rows')==len(rows_expected) and len(doc.get('results') or [])==len(rows_expected)), ('w2_inventory', isinstance(doc.get('w2'), dict) and sorted(doc['w2'])==['L1.00','L1.20','L1.50']), ('screen_sha', b.get('screen_sha256')==sc.get('screen_sha256') and isinstance(sc.get('n_blocking'), int)), ('campaign', isinstance(doc.get('campaign'), dict) and doc['campaign'].get('id')==lock['campaign_id'] and doc['campaign'].get('screen') is True), ('not_a_calibration', 'calibration' not in doc and 'per_pseudo_status' not in doc)]
                for k, ok in checks:
                    if not ok: evidence.append(f'published screen record not bound: {k}')
        REC.update(D4C1_SCREEN_COMPLETE=v.get('D4C1_SCREEN_COMPLETE'), record_sha256=sha(vp), record_stage=v.get('stage'), record_failures=v.get('failures'), gates=v.get('gates'), stages_rss_mb=v.get('stages_rss_mb'), stages_peak_rss_mb=v.get('stages_peak_rss_mb'), seconds_script=v.get('seconds'), timings=v.get('timings'), screen_summary={k: (v.get('screen') or {}).get(k) for k in ('n_rows','n_blocking','w2_applicable','w2','screen_sha256','rows','ratio_bound_max','stages_covered')})
        print('rc', rc.returncode, '| stage', v.get('stage'), '| D4C1_SCREEN_COMPLETE', v.get('D4C1_SCREEN_COMPLETE'), '| failures', v.get('failures'), '| peak rss MB', v.get('stages_peak_rss_mb'), '| timings', v.get('timings')); print('screen', REC['screen_summary'])
    REC.update(launcher_fallback=bool(shape), gates_ok=(not gates_bad) if v is not None else None, bindings_ok=(not binding) if v is not None else None, evidence_ok=(not evidence) if v is not None else None, stderr_tail=rc.stderr[-2000:])
    screen_complete=bool(rc.returncode==0 and v is not None and v.get('D4C1_SCREEN_COMPLETE') is True and not shape and not gates_bad and not binding and not evidence)      # the number of blocking rows is NOT a condition; the screen is not a calibration PASS
    _finish('complete', shape+gates_bad+binding+evidence)
except Exception as attempt_error:
    screen_complete=False; _finish(REC.get('stage') or 'unknown', [f'exception at stage {REC.get("stage")}: {attempt_error!r}'], attempt_error)
print('screen_complete', screen_complete)


staged in 132 s; disk free GB 213.2
  first-wave intake 20101 (L1.00) 14s rss 1753 MB
  first-wave intake 20102 (L1.00) 12s rss 2188 MB
  first-wave intake 20103 (L1.00) 12s rss 2486 MB
  first-wave intake 20201 (L1.20) 12s rss 2758 MB
  first-wave intake 20202 (L1.20) 12s rss 3053 MB
  first-wave intake 20203 (L1.20) 13s rss 3361 MB
  first-wave intake 20301 (L1.50) 13s rss 3644 MB
  first-wave intake 20302 (L1.50) 16s rss 3940 MB
  first-wave intake 20303 (L1.50) 22s rss 4224 MB
  screen: {"n_rows": 2000, "n_blocking": 2000, "w2_applicable": true}
run finalization | stage: complete | gates: {"G_pins_loaded": true, "G_engine_inventory": true, "G_script_sha": true, "G_phaseC_members": true, "G_env_lock": true, "G_external_loader_sha": true, "G_twelve_context": true, "G_w2_context_registered": true, "G_pseudo_registered": true, "G_twelve_asset_registered": true, "G_commitment_form": true, "G_inputs_resolved": true, "G_first_wave_supplies": true, "G_plans_fixed": true, "G_first_wave_view

In [4]:
# --- 3. zip the evidence (lock, attempt records, run record, screen record, stdout/stderr) for the audit and the certificate (--mode certificate runs anywhere: no Colab needed)
rt=(f"_r{lock['row_range'][0]:04d}_{lock['row_range'][1]:04d}" if lock['row_range'] else ''); zp=f'/content/d4c1_screen_{lock["family"]}{rt}_{REPO_COMMIT[:12]}.zip'; shutil.make_archive(zp[:-4], 'zip', root_dir=lock['out'])
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)


/content/d4c1_screen_E2_ba51e5951ebb.zip 548505


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>